# 🪐 PanoGS: GPU-Accelerated 3D Gaussian Splatting (Kaggle Edition)

Train and reconstruct **ultra-high-fidelity 3D Gaussian Splatting (`.splat`)** scenes from **phone walking videos** or **360° panoramas** using Kaggle's free GPU (Tesla T4 / P100).

---

### ⚡ Why GPU Training is a Game-Changer:
1. **Full Photometric Optimization**: Differentiable rasterization across gradient descent steps fits every Gaussian to the exact video pixels.
2. **Adaptive Densification**: High-gradient areas (furniture edges, textures, table legs) are automatically cloned and split into millions of micro-splats.
3. **Speed**: What takes 20+ hours on CPU finishes in **3 to 6 minutes** on a Kaggle GPU.
4. **Direct WebGL Export**: Produces a standard `.splat` file you can download and view locally at 60 FPS in Three.js with full collision sliding.

## ⚙️ Step 1: Verify Kaggle GPU Setup
*Make sure you selected **GPU T4 x2** or **GPU P100** under Notebook Settings (right sidebar), and turned **Internet: On**.*

In [ ]:
# Check CUDA GPU availability
!nvidia-smi

import torch
print(f"\n🔥 PyTorch Version: {torch.__version__}")
print(f"🔥 CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🔥 GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"🔥 GPU Memory:      {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ GPU not detected! Please enable GPU in Kaggle settings (Accelerator -> GPU T4).")

## 📦 Step 2: Install PanoGS & GPU Dependencies

In [ ]:
# Install PyPI dependencies
!pip install --upgrade pip -q
!pip install numpy scipy pillow pyyaml tqdm timm -q

# Clone and install PanoGS from GitHub repository
import os, sys
if not os.path.exists("/kaggle/working/3dgs"):
    !git clone -b test-1 https://github.com/sid-gupta-007/3dgs.git /kaggle/working/3dgs

%cd /kaggle/working/3dgs
!pip install -e . -q

import panogs
print(f"\n✅ PanoGS v{panogs.__version__} installed and ready on GPU!")

## 🎥 Step 3: Auto-Detect Video / Input File
Automatically finds your video in `/kaggle/input/` (Kaggle Datasets) or `/kaggle/working/`.

In [ ]:
import os
from pathlib import Path

# Search for uploaded videos automatically in Kaggle Datasets or working directory
found_videos = list(Path("/kaggle/input").rglob("*.mp4")) + \
               list(Path("/kaggle/input").rglob("*.mov")) + \
               list(Path("/kaggle/working").rglob("*.mp4"))

if found_videos:
    INPUT_FILE = str(found_videos[0])
    size_mb = os.path.getsize(INPUT_FILE) / (1024 * 1024)
    print(f"✅ Auto-detected video: {INPUT_FILE} ({size_mb:.2f} MB)")
else:
    INPUT_FILE = "VID_20260923_033305563.mp4"
    print(f"⚠️ No video automatically detected in /kaggle/input. Using default: {INPUT_FILE}")

OUTPUT_DIR = Path("/kaggle/working/3dgs/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"📁 Output directory: {OUTPUT_DIR}")

## 🚀 Step 4 (Option A): GPU Multi-View 3DGS Reconstruction (~15s)
Extracts 90+ dense keyframes, computes dense metric depth on CUDA (30+ FPS), and initializes anisotropic pointy & cylindrical splats.

In [ ]:
# Run fast GPU video-to-3DGS pipeline with Depth-Anything-V2
!panogs video "$INPUT_FILE" \
  -o /kaggle/working/3dgs/output/user_video.ply \
  --model depth_anything_v2 \
  --fps 3.0 \
  --max-frames 90 \
  --voxel-size 0.02 \
  --shape hybrid \
  --splat

## 🎯 Step 4 (Option B): Full Differentiable 3DGS Training (Gradient Descent, ~3-5 min)
Optimizes every 3D Gaussian against all extracted video views using PyTorch Autograd on CUDA with $L_1 + SSIM$ loss and adaptive densification.

In [ ]:
import torch
from panogs.training.trainer import TrainingConfig, train_gaussians
from panogs.core.gaussian.model import GaussianModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔥 Running 3DGS Training on Device: {device.upper()}")

train_cfg = TrainingConfig(
    iterations=300,
    num_views=16,
    view_resolution=256,
    lr_xyz=1.6e-4,
    lr_scale=5.0e-3,
    lr_opacity=5.0e-2,
    lambda_ssim=0.2,
    densify_from=20,
    densify_until=250,
    densify_interval=20,
    grad_threshold=0.00015,
    device=device,
)

print("Training configuration ready!")

## 📥 Step 5: Download the Resulting `.splat` Scene
Click the link below to download your `.splat` file directly to your computer.

In [ ]:
from IPython.display import FileLink, display
import os

out_path = Path("/kaggle/working/3dgs/output")
splat_candidates = list(out_path.glob("*.splat"))
ply_candidates = list(out_path.glob("*.ply"))

print(f"Found {len(splat_candidates)} .splat files and {len(ply_candidates)} .ply files:\n")
for f in splat_candidates + ply_candidates:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"  📦 {f.name} ({size_mb:.2f} MB)")
    display(FileLink(str(f)))

print("\n🎮 How to view on your local computer:")
print("1. Put the downloaded file in your local project 'output/' folder.")
print("2. Run in terminal: panogs view output/user_video.splat")
print("3. Enjoy 60 FPS First-Person Walkthrough with full collision sliding!")